In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv("DateFruit_Dataset.csv")


In [3]:
df.head() #34 features, 1 output 

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [4]:
df.shape

(898, 35)

In [5]:
X=df.drop("Class",axis=1)
Y=df["Class"]

In [6]:
df["Class"].unique()

<StringArray>
['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY']
Length: 7, dtype: str

In [7]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

le=LabelEncoder()
Y=le.fit_transform(Y)

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test=train_test_split(X,Y,test_size=0.2,random_state=42)

In [9]:
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [11]:
X_train_tensor=torch.tensor(X_train_scaled,dtype=torch.float32)
Y_train_tensor=torch.tensor(Y_train,dtype=torch.long)

X_test_tensor=torch.tensor(X_test_scaled,dtype=torch.float32)
Y_test_tensor=torch.tensor(Y_test,dtype=torch.long)

In [12]:
train_dataset=TensorDataset(X_train_tensor,Y_train_tensor)
test_dataset=TensorDataset(X_test_tensor,Y_test_tensor)

In [15]:
train_loader=DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader=DataLoader(test_dataset, batch_size=32, shuffle=True)

In [25]:
#Build Model

class ANN(nn.Module):
    def __init__(self):
        super(ANN,self).__init__()

        self.model=nn.Sequential(

            #hidden layer 1
            nn.Linear(X.shape[1],64),
            nn.ReLU(),
            #hidden layer 2
            nn.Linear(64,64),
            nn.ReLU(),
            #output layer
            nn.Linear(64,7)
        )
    def forward(self,X):
            return self.model(X)

In [26]:
model=ANN()

# loss and optim
criteria=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

In [27]:
# Training the NN

epochs=100

for epoch in range(epochs):
    model.train()

    running_loss=0.0
    

    for xb, yb in train_loader:
        optimizer.zero_grad()
        outputs=model(xb)
        loss=criteria(outputs, yb)
        loss.backward()
        optimizer.step() #weights or parameters gets uppdated

        running_loss+=loss.item()

    train_loss=running_loss/len(train_loader)

    print(f"epoch={epoch}/{epochs}, loss={train_loss}")
        

epoch=0/100, loss=1.706316538479017
epoch=1/100, loss=1.0805098414421082
epoch=2/100, loss=0.6836140350155209
epoch=3/100, loss=0.5162683362546174
epoch=4/100, loss=0.43985769670942554
epoch=5/100, loss=0.38250174211419147
epoch=6/100, loss=0.3406013256829718
epoch=7/100, loss=0.32084407624991046
epoch=8/100, loss=0.30352608276450116
epoch=9/100, loss=0.2713801880245623
epoch=10/100, loss=0.26124211012021353
epoch=11/100, loss=0.24506212997695673
epoch=12/100, loss=0.22809362119954565
epoch=13/100, loss=0.2138432607702587
epoch=14/100, loss=0.2062536957471267
epoch=15/100, loss=0.20494040215144987
epoch=16/100, loss=0.19588713607062463
epoch=17/100, loss=0.1829616103483283
epoch=18/100, loss=0.16769180932770605
epoch=19/100, loss=0.1719367649892102
epoch=20/100, loss=0.1568297157469003
epoch=21/100, loss=0.1531487231993157
epoch=22/100, loss=0.14641351479551065
epoch=23/100, loss=0.14257679620514746
epoch=24/100, loss=0.13695035421329996
epoch=25/100, loss=0.13233552354833353
epoch=26/

In [34]:
model.eval()

total=0
correct=0

with torch.no_grad():
    for xb, yb in test_loader:
        outputs=model(xb)
        _, predicted=torch.max(outputs,1) #torch.max() returns the maximum value in the tensor and as well as its index, here index indicates the predicted class category
        correct+=(predicted==yb).sum().item()
        total+=yb.size(0) #actual samples in each batch 

print("Total Values",total)
print("Correct Values",correct)
print("Accuracy of the model",(correct/total) *100)




Total Values 180
Correct Values 171
Accuracy of the model 95.0
